# HelioCloud Storage + Burst + SDO

* HelioCloud is an AWS Cloud environment that is user-friendly
* Daskhub is a cloud Notebooks setup that allows parallel processing via Dask
* Dask lets you temporarily throw lots of CPUs at a problem
* S3 is the big cheap AWS storage
* SDO is a mission with lots of image data

Here we combine a few HelioCloud demos with our big data fileRegistry to tackle 1 year of SDO data.  The data is in AWS S3 and we do not copy the files over, but instead have CPUs at AWS access it directly

* 1 year of SDO 94A EUV images from AIA is 129,758 files, each 14MB, totalling 1.8 TB.
* This code calculates a simple irradiance
* If done serially on your laptop, it would take 27 hours
* HelioCloud takes 25 minutes (1467 sec) to analyze through 1 year of SDO data

(More fun stats: that's 88 files/second, also 1 GB/sec to do the full analysis (which is 2x the read speed of a SATA SSD.  It is 2x faster than it would take to just copy the files off a local disk and 8x faster than copying via gigabit internet).


In [1]:
import cloudcatalog
import boto3
import dask
import io
import logging
import time
import re
import pickle
import numpy as np
from astropy.io import fits
import astropy.io.fits
from dask.distributed import Client
from dask_gateway import Gateway, GatewayCluster
import math
import dask.bag as db
import s3fs
from dask.distributed import get_worker

In [2]:
fr=cloudcatalog.CloudCatalog("s3://gov-nasa-hdrl-data1/")
frID = "aia_0094"
start, stop = '2020-01-01T00:00:00Z', '2020-12-31T23:59:59Z'
file_registry1 = fr.request_cloud_catalog(frID, start_date=start, stop_date=stop, overwrite=False)
filelist = file_registry1['datakey'].to_list()

In [3]:
testing = False
if testing:
    s3_files = filelist[0:1000] # small test set to test
else:
    s3_files = filelist
print(len(s3_files))

129758


In [ ]:
# number of workers to use, for automatic scaling, our max number
n_workers = 5 # 10-50 works well, more workers actually went slower
# memory per worker (in Gb), typically 1, 2 or 4GB
w_memory = 2
# cores per worker, must be 1-4
w_cores = 2

# from the daskhub tutorial, setting up dask
gateway = Gateway()
options = gateway.cluster_options()
options.worker_cores = w_cores
options.worker_memory = w_memory

# initialize cluster and create client, takes < 15 seconds

cluster = gateway.new_cluster(options)
client = cluster.get_client() # can also use 'client=Client(cluster)'
#cluster.adapt(minimum=10, maximum=n_workers)
cluster.scale(n_workers)

# This calls the widget
cluster

## Using worker plugin

In [ ]:
from dask.distributed import WorkerPlugin, get_worker

class S3FSPlugin(WorkerPlugin):
    def __init__(self, max_conns=128):
        self.max_conns = max_conns

    def setup(self, worker):
        import s3fs
        # one S3FileSystem per worker process
        if not hasattr(worker, "s3"):
            worker.s3 = s3fs.S3FileSystem(
                anon=False,
                default_fill_cache=False,
                config_kwargs={"max_pool_connections": self.max_conns},
            )

# Register once after creating your client; applies to existing and future workers
client.register_plugin(S3FSPlugin(128), name="s3fs")

In [ ]:
def get_irradiance(arr):
    # fast path: mean on the raw image array
    return float(arr.mean())
    
# --- single-file processing (no prints inside tasks) ---
def process_one_file(fs, s3url):
    # returns a dict
    # s3fs provides a seekable, streaming file-like object
    with fs.open(s3url, mode="rb") as f:
        with fits.open(f, memmap=False) as hdul:
            hdu = hdul[1]                 # adjust if your image HDU differs
            date = hdu.header.get("T_OBS")
            irrad = get_irradiance(hdu.data)
            return {"t_obs": date, "irradiance": irrad, "s3url": s3url}

In [ ]:
def process_partition(urls):
    w = get_worker()
    fs = w.s3  # set by the plugin
    out = []
    for u in urls:
        row = process_one_file(fs, u)
        if row is not None:
            out.append(row)
    return out

## Without Worker Plugin

In [ ]:
def get_irradiance(arr):
    # fast path: mean on the raw image array
    return float(arr.mean())
    
# --- single-file processing (no prints inside tasks) ---
def process_one_file(fs, s3url):
    # returns a dict
    # s3fs provides a seekable, streaming file-like object
    with fs.open(s3url, mode="rb") as f:
        with fits.open(f, memmap=False) as hdul:
            hdu = hdul[1]                 # adjust if your image HDU differs
            date = hdu.header.get("T_OBS")
            irrad = get_irradiance(hdu.data)
            return {"t_obs": date, "irradiance": irrad, "s3url": s3url}

def process_partition(urls):
    fs = s3fs.S3FileSystem(
            anon=False,
            default_fill_cache=False,
            config_kwargs={"max_pool_connections": 128},
            )

    out = []
    for u in urls:
        row = process_one_file(fs, u)
        if row is not None:
            out.append(row)
    return out

## Send job to client

In [ ]:
#10,000 files done in 360 s with 10 partitions (20 workers, 2 cores, scaled)
#10,000 files done in 519 s with 20 partitions (20 workers, 2 cores, scaled)
#10,000 files done in 510 s with 5 partitions (20 workers, 2 cores, scaled)
#10,000 files done in 514 s with 10 partitions (20 workers, 2 cores, scaled) (likely cluster had lost workers)
#10,000 files done in 360s with 10 partitions (20 workers, 2 cores, scaled)
#10,000 files done in 348 seconds with 20 partitions (20 worksers, 2 cores, scaled)
#10,000 files done in 357 seconds with 50 partitions (20 workers, 2 cores, scaled)
#10,000 files done in 344 seconds with 5 partitions (20 workers, 2 cores, scaled)
#10,000 files done in 346 seconds with 1 partition (20 workers, 2 cores, scaled)
#10,000 files done in 182 seconds with 5 partitions (40 workers, 2 cores, scaled)
#100,000 files done in 1775.1 seconds with 5 partitions (40 workers, 2 cores, scaled)
#100,000 files done in 1596.7 seconds with 5 partitions (40 workers, 4 cores, scaled)

In [ ]:
s3_files = filelist[0:1000] # small test set to test
print(len(s3_files))

PARTITION_SIZE = 50
print(PARTITION_SIZE)

b = db.from_sequence(s3_files, partition_size=PARTITION_SIZE)

t0 = time.time()
results = b.map_partitions(process_partition).compute()
elapsed = time.time() - t0

print(f"Processed {len(results)} records from len(s3_files) files in {elapsed:.1f}s")
# results is: [{"t_obs": "...", "irradiance": 123.45, "s3url": "s3://..."}, ...]


In [ ]:
sample = s3_files[:100]
b = db.from_sequence(sample, partition_size=100)
start = time.time()
n = (b.map_partitions(process_partition).flatten().count().compute())
sec_per_file = (time.time()-start) / max(n, 1)
target_task_sec = 4.0     # aim 2–10s tasks
PARTITION_SIZE = max(25, int(target_task_sec / max(sec_per_file, 1e-3)))
print(f"Estimated {sec_per_file:.3f} s/file -> PARTITION_SIZE ≈ {PARTITION_SIZE}")

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df.head()

In [ ]:
irradiances = [r["irradiance"] for r in results if r.get("irradiance") is not None]

In [ ]:
irradiances[::100]

In [ ]:
if isinstance(results, list):
    # list of dicts — expected from the compute()
    print(results[:3])

In [ ]:
parts = b.map_partitions(process_partition).compute()
print(type(parts), len(parts))
print("Partition types:", {type(p) for p in parts})
# Check first few partitions
for i, p in enumerate(parts[:5]):
    print(i, type(p), (list(p)[:1] if hasattr(p, '__iter__') and not isinstance(p, (bytes, str)) else p))

In [ ]:
cluster.shutdown()

# Actual Analysis code

Here's our analysis routines, that operate on a single FITS file and returns the results of our 'work_on_data()' function.

In [ ]:
def DO_SCIENCE(mydata):
    # you can put better science here
    iirad = mydata.mean()
    return iirad

# these are variable helpful handler functions
def s3url_to_bucketkey(s3url: str): # -> Tuple[str, str]:
    """
    Extracts the S3 bucket name and file key from an S3 URL.
    e.g. s3://mybucket/mykeypart1/mykeypart2/fname.fits -> mybucket, mykeypart1/mykeypart2/fname.fits
    """
    name2 = re.sub(r"s3://","",s3url)
    s = name2.split("/",1)
    return s[0], s[1]

def process_fits_s3(s3key:str): # -> Tuple[str, float]:
    """ For a single FITS file, read it from S3, grab the header and
        data, then do the DO_SCIENCE() call of choice
    """
    sess = boto3.session.Session() # do this each open to avoid thread problem 'credential_provider'
    s3c = sess.client("s3")
    mybucket,mykey = s3url_to_bucketkey(s3key)
    try:
        fobj = s3c.get_object(Bucket=mybucket,Key=mykey)
        rawdata = fobj['Body'].read()
        bdata = io.BytesIO(rawdata)
        hdul = astropy.io.fits.open(bdata,memmap=False)        
        date = hdul[1].header['T_OBS']
        irrad = DO_SCIENCE(hdul[1].data)
    except:
        print("Error fetching ",s3key)
        date, irrad = None, None
        
    return date, irrad


# Accessing S3 from anywhere
The power of cloud is moving compute (CPUs) to the data, instead of hauling data over internet.  But if you want to play, here is a serial example of fetching the first 10 SDO files and processing them anywhere.  It is serial and slow-- roughly 66x slower than using parallel processing and 128x or more slow if you are fetching from S3 to your local machine.

In [ ]:
# Serial test it on the first ten files, so slow
now=time.time()
for i in range(10):
    results = process_fits_s3(filelist[i])
    print(results)
print(time.time()-now)

# Setting up the Dask 'burst' Configuration
This is pretty standard daskhub configuration from the HelioCloud dask demos, that works to burst tasks into the cloud.

We default to setting testing as 'True' so it only runs on 1,000 SDO files. If you set it to 'False', it will go through all the files.

In [ ]:
testing = True
if testing:
    s3_files = filelist[0:10000] # small test set to test
else:
    s3_files = filelist
print(len(s3_files))

Here are some Dask parameters for setting up your run.  These are non-optimized values, feel free to play with them to improve performance.

In [ ]:
# number of workers to use, for automatic scaling, our max number
n_workers = 10 # 10-50 works well, more workers actually went slower
# memory per worker (in Gb), typically 1, 2 or 4GB
w_memory = 4
# cores per worker, must be 1-4
w_cores = 4
# use Manual (if False, then uses Automatic scaling)
use_manual_scaling = False

useGUI = False # set true if you want the Dask widget as well

Now we initialzie the Dask gateway and cluster, using your above parameters, to set up the virtual machines that will subsequently operate on the data.

In [ ]:
# from the daskhub tutorial, setting up dask
gateway = Gateway()
options = gateway.cluster_options()
options.worker_cores = w_cores
options.worker_memory = w_memory

In [ ]:
# initialize cluster and create client, takes < 15 seconds

cluster = gateway.new_cluster(options)
client = cluster.get_client() # can also use 'client=Client(cluster)'
cluster.adapt(minimum=1, maximum=n_workers)

# This calls the widget
if useGUI: cluster

In [ ]:
client # let us take a look at it

# Now the actual algorithm work and runtime

In this case, our earlier 'file_registry1' is our set of fully qualified s3:// objects.  This code takes around 5 minutes for 10,000 files and 25 minutes for the full 130K files.

In [ ]:
now=time.time()

# simple version step 1, do it
time_irrad = client.map(process_fits_s3, s3_files)

# optional spot-check that jobs were sent
print(time_irrad[0:4])

# simple version step 2, gather results
all_data = client.gather(time_irrad)

print("Done! Completed in time ",(time.time()-now)/60.0,"minutes, on",len(all_data),"files")

Remember to always shut down your cluster (so you are not spending $ on unused resources.)

# Done, now do something with the results
Time to save your results, plot it, and carry on with your analysis.

In [ ]:
# let us save this
with open('SDO_test-big.pickle','wb') as fout:
    pickle.dump(all_data, fout)

In [ ]:
# Goodness check-- did any files not process?
a = [a for a in all_data if a[0] is None]
print("Bad fields: ",len(a))
plotme = [a for a in all_data if a[0] is not None]
print("Good fields: ",len(plotme))
#We're going to plot only every 100th point to save time
plotme = plotme[::100]

In [ ]:
from matplotlib import pyplot as plt
from matplotlib import dates as mdates
%matplotlib inline 
# Have Matplotlib create vector (svg) instead of raster (png) images
#%config InlineBackend.figure_formats = ['svg'] 
#plt.figure()
dates, values = zip(*plotme)
plt.plot(dates, values) #, marker='o', linestyle='-')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=90)
plt.show()

In [ ]:
# always shutdown
yn=input("Ready to shut down the cluster? y/n: ")
if yn.startswith('y'):
    cluster.shutdown()

In [ ]:
import dask
print(dask.__version__)

import distributed
print(distributed.__version__)

import dask_gateway
print(dask_gateway.__version__)